# 2D and 3D spheroid workflow

Inspect a pixel-based configuration, measure nuclei and spheroids, then build radial and exported feature tables.

In [ ]:
import json
import numpy as np
from pathlib import Path
from nuclear_spheroid_analysis import analyze_radial_2d, measure_spheroid_system
root = Path.cwd()
while not (root / '2D_nuclear_spheroid_master').exists():
    root = root.parent
configs_2d = sorted((root / '2D_nuclear_spheroid_master/config_files').glob('*.json'))
configs_3d = sorted((root / '3D_nuclear_spheroid_master/config_files').glob('*.json'))
len(configs_2d), len(configs_3d)


In [ ]:
example = configs_2d[0]
config = json.loads(example.read_text())
example.name, config


In [ ]:
yy, xx = np.ogrid[:64, :64]
spheroid_labels = (((yy - 32) ** 2 + (xx - 32) ** 2) <= 24 ** 2).astype(np.uint16)
nuclear_labels = np.zeros((64, 64), dtype=np.uint16)
nuclear_labels[20:24, 20:24] = 1
nuclear_labels[30:35, 30:35] = 2
nuclear_labels[43:47, 38:42] = 3
nuclear_image = nuclear_labels.astype(np.float32) / nuclear_labels.max()
nuclei, spheroids = measure_spheroid_system(
    nuclear_image, nuclear_labels, spheroid_labels,
)
nuclei[['label-id', 'of-spheroid', 'in-spheroid']], spheroids[['label-id', 'nuclei-count']]


In [ ]:
nuclei['DAPI_intensity-mean'] = nuclei['intensity-mean']
radial, nuclei, spheroids = analyze_radial_2d(
    nuclei, spheroids, nuclear_labels, spheroid_labels,
    {'nbins-sph': 4, 'maxdist-sph-shell': 24, 'maxdist-sph-outward': 24},
    {},
)
radial


In [ ]:
output_directory = Path('outputs')
output_directory.mkdir(exist_ok=True)
nuclei.to_csv(output_directory / 'nuclear_features.csv', index=False)
spheroids.to_csv(output_directory / 'spheroid_features.csv', index=False)
radial.to_csv(output_directory / 'spheroid_radial_features.csv', index=False)
